# Business Entity Resolution — Memory-Safe, Chunked Pipeline (v2)

This is a rewrite of the earlier version specifically to stop Colab from running out of RAM and crashing.
The fix is architectural, not just "use less data":

- **Nothing builds one giant all-pairs DataFrame.** Source-1 entities are processed in **chunks** (default 10,000
  at a time). For each chunk we generate candidates, compute features, and (at inference time) **write results
  straight to disk and discard the chunk** before moving to the next one. Peak RAM stays roughly constant no
  matter how many millions of Source-1 rows there are.
- **No unused columns.** The old version computed an address-token `frozenset` for every single row of every
  source and never used it. Gone.
- **Token sets are computed on the fly and discarded**, not stored as a permanent column across 5M+ row sources.
- **Training uses a capped random sample of Source-1 entities** (`MAX_TRAIN_S1`, default 150,000) rather than
  all ~2.2M. This is a deliberate, sane ML choice, not a shortcut that hurts you: with only ~10 features, a
  gradient boosted model saturates in quality long before you'd need every one of 2.2M training entities. The
  **test set is still scored for every single entity** — the cap only affects how much data trains the model.
- The blocking index's postings lists are stored as `array('i', ...)` (compact C-level integer arrays) instead
  of plain Python lists of ints — several times smaller in memory for the same content.

**If it still crashes:** lower `CHUNK_SIZE` and/or `MAX_TRAIN_S1` in the config cell (both are the main memory
knobs), and use Runtime → Change runtime type → **High-RAM** if you have Colab Pro/Pro+. Also do **Runtime →
Restart runtime** before re-running, so a previous crashed run's leaked memory doesn't carry over.


In [ ]:
# ---- 1. Setup ----
!pip install -q rapidfuzz


In [ ]:
import re, os, gc, time
from array import array
from collections import defaultdict
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

pd.set_option('display.max_colwidth', 80)

# ====== CONFIG ======
DATA_DIR = '/content'
OUT_DIR  = '/content/output'
os.makedirs(OUT_DIR, exist_ok=True)

SAMPLE_MODE = True        # True = quick dev run on a slice of Source-1. Set False for the real run.
SAMPLE_N_S1 = 3000        # only used when SAMPLE_MODE=True

# ---- the memory knobs: lower these first if you still hit RAM crashes ----
CHUNK_SIZE       = 10000    # how many Source-1 rows are processed (candidates+features+write) per batch
MAX_TRAIN_S1     = 150000   # cap on how many Source-1 training entities are used to build the model
                             # (full test set is ALWAYS scored in full -- this only caps training data)

TOPK_CANDIDATES   = 40
MIN_TOKEN_LEN     = 3
MAX_POSTING_LEN   = 4000
RANDOM_STATE      = 42
VAL_FRACTION      = 0.15

print('Config set. SAMPLE_MODE =', SAMPLE_MODE, '| CHUNK_SIZE =', CHUNK_SIZE, '| MAX_TRAIN_S1 =', MAX_TRAIN_S1)


In [ ]:
# ---- 2. Load data ----
def load_tsv(name, nrows=None):
    path = os.path.join(DATA_DIR, name)
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, nrows=nrows)

if SAMPLE_MODE:
    train_s1_full = load_tsv('train_source1.tsv', nrows=SAMPLE_N_S1)
    gt_full  = load_tsv('train_ground_truth.tsv')
    train_gt = gt_full[gt_full['source1_entity_id'].isin(train_s1_full['entity_id'])].reset_index(drop=True)
else:
    train_s1_full = load_tsv('train_source1.tsv')
    train_gt = load_tsv('train_ground_truth.tsv')

train_s2 = load_tsv('train_source2.tsv')
train_s3 = load_tsv('train_source3.tsv')

test_s1 = load_tsv('test_source1.tsv', nrows=SAMPLE_N_S1 if SAMPLE_MODE else None)
test_s2 = load_tsv('test_source2.tsv')
test_s3 = load_tsv('test_source3.tsv')

print('train_s1_full:', train_s1_full.shape, '| train_s2:', train_s2.shape, '| train_s3:', train_s3.shape)
print('test_s1:', test_s1.shape, '| test_s2:', test_s2.shape, '| test_s3:', test_s3.shape)
if SAMPLE_MODE:
    del gt_full
gc.collect()


In [ ]:
# ---- 3. Normalization (lean: only the 4 columns we actually use later, no stored token sets) ----
LEGAL_SUFFIXES = {
    'inc','incorporated','llc','l.l.c','ltd','limited','corp','corporation','co','company',
    'pvt','private','pllc','llp','plc','lp','gmbh','sarl','sa','pc','p.c','co.', 'group','holdings',
    'the','and','of'
}
_non_alnum = re.compile(r'[^a-z0-9\s]')
_ws = re.compile(r'\s+')

def normalize_text(s):
    if s is None:
        return ''
    s = str(s).lower().strip()
    s = _non_alnum.sub(' ', s)
    s = _ws.sub(' ', s).strip()
    return s

def tokens_from_norm(norm_s):
    # recomputed on the fly from an already-normalized string -- cheap, so we never need to
    # permanently store a frozenset/list column for millions of rows.
    if not norm_s:
        return []
    return [t for t in norm_s.split() if t and t not in LEGAL_SUFFIXES]

def extract_zip(addr):
    digits = re.findall(r'\d{4,}', str(addr))
    return max(digits, key=len) if digits else ''

def prep_source(df):
    out = pd.DataFrame({'entity_id': df['entity_id'].values})
    out['norm_name']    = df['business_name'].map(normalize_text)
    out['norm_addr']    = df['business_address'].map(normalize_text)
    out['zip']          = df['business_address'].map(extract_zip)
    out['country_norm'] = df['country'].map(normalize_text)
    return out

train_s1_full = prep_source(train_s1_full)
train_s2      = prep_source(train_s2)
train_s3      = prep_source(train_s3)
test_s1       = prep_source(test_s1)
test_s2       = prep_source(test_s2)
test_s3       = prep_source(test_s3)
gc.collect()
print('normalization done (4 lean columns per source, no stored token sets)')
train_s1_full.head()


In [ ]:
# ---- 4. Blocking index: country-partitioned inverted index, compact array('i') postings ----
def build_index(df):
    raw = defaultdict(list)   # temp: python lists while building
    names = df['norm_name'].values
    zips  = df['zip'].values
    countries = df['country_norm'].values
    for pos in range(len(df)):
        c = countries[pos]
        for t in set(tokens_from_norm(names[pos])):
            if len(t) >= MIN_TOKEN_LEN:
                raw[(c, 'T', t)].append(pos)
                raw[(c, 'P', t[:4])].append(pos)
        z = zips[pos]
        if z:
            raw[(c, 'Z', z)].append(pos)
    # drop over-common keys, then freeze remaining postings into compact int32 arrays
    idx = {}
    for k, v in raw.items():
        if len(v) <= MAX_POSTING_LEN:
            idx[k] = array('i', v)
    raw.clear(); del raw
    gc.collect()
    return idx

def get_candidates(name_norm, country, zip_, idx, topk=TOPK_CANDIDATES):
    counts = defaultdict(int)
    for t in set(tokens_from_norm(name_norm)):
        if len(t) >= MIN_TOKEN_LEN:
            for pos in idx.get((country, 'T', t), ()):
                counts[pos] += 2
            for pos in idx.get((country, 'P', t[:4]), ()):
                counts[pos] += 1
    if zip_:
        for pos in idx.get((country, 'Z', zip_), ()):
            counts[pos] += 3
    if not counts:
        return []
    return sorted(counts.items(), key=lambda kv: -kv[1])[:topk]

t0 = time.time()
idx2 = build_index(train_s2)
idx3 = build_index(train_s3)
print(f'train indexes built in {time.time()-t0:.1f}s | keys: idx2={len(idx2)}, idx3={len(idx3)}')


In [ ]:
# ---- 5. Chunked candidate+feature generation (shared by training-collection and inference) ----
FEATURE_COLS = ['block_score','name_ratio','name_token_sort','name_token_set','name_partial',
                'addr_token_sort','addr_partial','country_match','zip_match','name_len_diff']

def build_chunk_pairs(s1_chunk, s2_df, s3_df, idx2, idx3):
    """Given a small chunk of Source-1 rows, return a DataFrame of candidate pairs with features.
    All lookups are done via numpy .values indexing (no full-table merges), and everything here is
    discarded by the caller once processed -- nothing persists beyond one chunk."""
    s2_name = s2_df['norm_name'].values; s2_addr = s2_df['norm_addr'].values
    s2_country = s2_df['country_norm'].values; s2_zip = s2_df['zip'].values; s2_ids = s2_df['entity_id'].values
    s3_name = s3_df['norm_name'].values; s3_addr = s3_df['norm_addr'].values
    s3_country = s3_df['country_norm'].values; s3_zip = s3_df['zip'].values; s3_ids = s3_df['entity_id'].values

    s1_id_col, cand_id_col, name1_col, name2_col, addr1_col, addr2_col = [], [], [], [], [], []
    country1_col, country2_col, zip1_col, zip2_col, block_col = [], [], [], [], []

    for s1id, nm, ad, ctry, z in zip(s1_chunk['entity_id'].values, s1_chunk['norm_name'].values,
                                      s1_chunk['norm_addr'].values, s1_chunk['country_norm'].values,
                                      s1_chunk['zip'].values):
        for pos, score in get_candidates(nm, ctry, z, idx2):
            s1_id_col.append(s1id); cand_id_col.append(s2_ids[pos])
            name1_col.append(nm); name2_col.append(s2_name[pos])
            addr1_col.append(ad); addr2_col.append(s2_addr[pos])
            country1_col.append(ctry); country2_col.append(s2_country[pos])
            zip1_col.append(z); zip2_col.append(s2_zip[pos]); block_col.append(score)
        for pos, score in get_candidates(nm, ctry, z, idx3):
            s1_id_col.append(s1id); cand_id_col.append(s3_ids[pos])
            name1_col.append(nm); name2_col.append(s3_name[pos])
            addr1_col.append(ad); addr2_col.append(s3_addr[pos])
            country1_col.append(ctry); country2_col.append(s3_country[pos])
            zip1_col.append(z); zip2_col.append(s3_zip[pos]); block_col.append(score)

    if not s1_id_col:
        return pd.DataFrame(columns=['source1_entity_id','cand_id'] + FEATURE_COLS)

    out = pd.DataFrame({'source1_entity_id': s1_id_col, 'cand_id': cand_id_col})
    out['name_ratio']      = [fuzz.ratio(a, b) for a, b in zip(name1_col, name2_col)]
    out['name_token_sort'] = [fuzz.token_sort_ratio(a, b) for a, b in zip(name1_col, name2_col)]
    out['name_token_set']  = [fuzz.token_set_ratio(a, b) for a, b in zip(name1_col, name2_col)]
    out['name_partial']    = [fuzz.partial_ratio(a, b) for a, b in zip(name1_col, name2_col)]
    out['addr_token_sort'] = [fuzz.token_sort_ratio(a, b) for a, b in zip(addr1_col, addr2_col)]
    out['addr_partial']    = [fuzz.partial_ratio(a, b) for a, b in zip(addr1_col, addr2_col)]
    out['country_match']   = [1 if a == b else 0 for a, b in zip(country1_col, country2_col)]
    out['zip_match']       = [1 if (a and a == b) else 0 for a, b in zip(zip1_col, zip2_col)]
    out['name_len_diff']   = [abs(len(a) - len(b)) for a, b in zip(name1_col, name2_col)]
    out['block_score']     = block_col
    return out

def iter_chunks(df, chunk_size):
    for i in range(0, len(df), chunk_size):
        yield df.iloc[i:i+chunk_size]


In [ ]:
# ---- 6. Build the (capped, sampled) training set chunk by chunk ----
gt_map = defaultdict(set)
for s1id, matches in zip(train_gt['source1_entity_id'], train_gt['matched_entity_ids']):
    if matches:
        gt_map[s1id] = set(matches.split(','))

rng = np.random.RandomState(RANDOM_STATE)
s1_ids_all = np.array(train_s1_full['entity_id'].tolist(), dtype=object)
rng.shuffle(s1_ids_all)
train_s1_sample_ids = set(s1_ids_all[:MAX_TRAIN_S1])
train_s1 = train_s1_full[train_s1_full['entity_id'].isin(train_s1_sample_ids)].reset_index(drop=True)
print(f'training on a sample of {len(train_s1)} / {len(train_s1_full)} Source-1 entities')

feat_chunks, id_chunks, cand_id_chunks, label_chunks = [], [], [], []
t0 = time.time()
for i, chunk in enumerate(iter_chunks(train_s1, CHUNK_SIZE)):
    pairs = build_chunk_pairs(chunk, train_s2, train_s3, idx2, idx3)
    if len(pairs):
        labels = np.array([1 if cid in gt_map.get(s1id, ()) else 0
                            for s1id, cid in zip(pairs['source1_entity_id'], pairs['cand_id'])], dtype=np.int8)
        feat_chunks.append(pairs[FEATURE_COLS].to_numpy(dtype=np.float32))
        id_chunks.append(pairs['source1_entity_id'].to_numpy(dtype=object))
        cand_id_chunks.append(pairs['cand_id'].to_numpy(dtype=object))
        label_chunks.append(labels)
    del pairs
    if i % 5 == 0:
        print(f'  train chunk {i} ({(i+1)*CHUNK_SIZE} rows) done, {time.time()-t0:.1f}s elapsed')
        gc.collect()

X_all = np.vstack(feat_chunks)
ids_all = np.concatenate(id_chunks)
cand_ids_all = np.concatenate(cand_id_chunks)
y_all = np.concatenate(label_chunks)
del feat_chunks, id_chunks, cand_id_chunks, label_chunks
gc.collect()
print('total labeled training pairs:', X_all.shape, '| positives:', y_all.sum())


In [ ]:
# ---- 7. Entity-level train/val split, train the classifier ----
from sklearn.ensemble import HistGradientBoostingClassifier

val_ids_set = set(np.array(list(train_s1_sample_ids))[
    rng.permutation(len(train_s1_sample_ids))[:max(1, int(len(train_s1_sample_ids)*VAL_FRACTION))]
])
is_val = np.array([sid in val_ids_set for sid in ids_all])

X_tr, y_tr = X_all[~is_val], y_all[~is_val]
X_va, y_va = X_all[is_val],  y_all[is_val]
ids_va, cand_va = ids_all[is_val], cand_ids_all[is_val]
print('train pairs:', X_tr.shape, '| val pairs:', X_va.shape)

clf = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.08, max_depth=6, l2_regularization=1.0,
    class_weight='balanced', random_state=RANDOM_STATE,
)
clf.fit(X_tr, y_tr)
print('trained. train acc:', clf.score(X_tr, y_tr), '| val acc:', clf.score(X_va, y_va))


In [ ]:
# ---- 8. Macro F0.5 + threshold tuning on the validation split ----
def f_beta(precision, recall, beta=0.5):
    if precision == 0 and recall == 0:
        return 0.0
    b2 = beta * beta
    denom = b2 * precision + recall
    return 0.0 if denom == 0 else (1 + b2) * precision * recall / denom

def macro_f05(pred_map, truth_map, all_ids):
    scores = []
    for sid in all_ids:
        pred, truth = pred_map.get(sid, set()), truth_map.get(sid, set())
        if not pred and not truth:
            scores.append(1.0); continue
        if not pred and truth:
            scores.append(0.0); continue
        tp = len(pred & truth)
        precision = tp/len(pred) if pred else 0.0
        recall = tp/len(truth) if truth else 0.0
        scores.append(f_beta(precision, recall, 0.5))
    return float(np.mean(scores))

va_prob = clf.predict_proba(X_va)[:, 1]
truth_val = {sid: gt_map.get(sid, set()) for sid in val_ids_set}

best_thr, best_score = 0.5, -1.0
for thr in np.arange(0.05, 0.96, 0.02):
    pred_map = defaultdict(set)
    keep = va_prob >= thr
    for sid, cid in zip(ids_va[keep], cand_va[keep]):
        pred_map[sid].add(cid)
    score = macro_f05(pred_map, truth_val, val_ids_set)
    if score > best_score:
        best_score, best_thr = score, thr
print(f'Best validation macro F0.5 = {best_score:.4f} at threshold = {best_thr:.2f}')

cand_by_s1 = defaultdict(set)
for sid, cid in zip(ids_va, cand_va):
    cand_by_s1[sid].add(cid)
hit = tot = 0
for sid in val_ids_set:
    truth = truth_val.get(sid, set())
    if not truth: continue
    tot += len(truth); hit += len(truth & cand_by_s1.get(sid, set()))
print(f'Blocking recall ceiling on validation: {hit}/{tot} = {hit/max(1,tot):.3f}')

del X_all, y_all, ids_all, cand_ids_all, X_tr, y_tr, X_va, y_va
gc.collect()


In [ ]:
# ---- 9. Chunked, streaming inference over the FULL test set -> write directly to disk ----
t0 = time.time()
tidx2 = build_index(test_s2)
tidx3 = build_index(test_s3)
print(f'test indexes built in {time.time()-t0:.1f}s')

valid_s2_ids = set(test_s2['entity_id']); valid_s3_ids = set(test_s3['entity_id'])

def dedupe_keep_order(ids):
    seen, out = set(), []
    for i in ids:
        if i not in seen:
            seen.add(i); out.append(i)
    return out

match_path = os.path.join(OUT_DIR, 'matching_results.tsv')
cand_path  = os.path.join(OUT_DIR, 'candidate_pairs.tsv')

with open(match_path, 'w') as fm, open(cand_path, 'w') as fc:
    fm.write('source1_entity_id\tmatched_entity_ids\n')
    fc.write('source1_entity_id\tcandidate_entity_ids\n')

t0 = time.time()
n_written = 0
for i, chunk in enumerate(iter_chunks(test_s1, CHUNK_SIZE)):
    pairs = build_chunk_pairs(chunk, test_s2, test_s3, tidx2, tidx3)
    if len(pairs):
        pairs['prob'] = clf.predict_proba(pairs[FEATURE_COLS].to_numpy(dtype=np.float32))[:, 1]
        by_s1_cand = defaultdict(list)
        by_s1_match = defaultdict(list)
        for sid, cid, prob in zip(pairs['source1_entity_id'], pairs['cand_id'], pairs['prob']):
            if cid in valid_s2_ids or cid in valid_s3_ids:
                by_s1_cand[sid].append(cid)
                if prob >= best_thr:
                    by_s1_match[sid].append(cid)
    else:
        by_s1_cand, by_s1_match = {}, {}

    match_lines, cand_lines = [], []
    for sid in chunk['entity_id'].values:
        cand_lines.append(f"{sid}\t{','.join(dedupe_keep_order(by_s1_cand.get(sid, [])))}\n")
        match_lines.append(f"{sid}\t{','.join(dedupe_keep_order(by_s1_match.get(sid, [])))}\n")

    with open(match_path, 'a') as fm, open(cand_path, 'a') as fc:
        fm.writelines(match_lines); fc.writelines(cand_lines)

    n_written += len(chunk)
    del pairs, by_s1_cand, by_s1_match, match_lines, cand_lines
    if i % 5 == 0:
        print(f'  test chunk {i}: {n_written}/{len(test_s1)} entities written, {time.time()-t0:.1f}s elapsed')
        gc.collect()

print('DONE. wrote', n_written, 'rows to', match_path, 'and', cand_path)


In [ ]:
# ---- 10. Local format validation ----
def validate(match_path, cand_path, test_s1_ids, valid_s2_ids, valid_s3_ids):
    issues = []
    m = pd.read_csv(match_path, sep='\t', dtype=str, keep_default_na=False)
    c = pd.read_csv(cand_path, sep='\t', dtype=str, keep_default_na=False)

    if list(m.columns) != ['source1_entity_id','matched_entity_ids']:
        issues.append(f'matching_results.tsv columns wrong: {list(m.columns)}')
    if list(c.columns) != ['source1_entity_id','candidate_entity_ids']:
        issues.append(f'candidate_pairs.tsv columns wrong: {list(c.columns)}')
    if m['source1_entity_id'].duplicated().any():
        issues.append('duplicate source1_entity_id rows in matching_results.tsv')
    if set(m['source1_entity_id']) != set(test_s1_ids):
        missing = set(test_s1_ids) - set(m['source1_entity_id'])
        extra = set(m['source1_entity_id']) - set(test_s1_ids)
        if missing: issues.append(f'{len(missing)} source1 entities missing from matching_results.tsv')
        if extra: issues.append(f'{len(extra)} unknown source1 ids in matching_results.tsv')

    cand_map = dict(zip(c['source1_entity_id'], c['candidate_entity_ids']))
    for s1id, ids_str in zip(m['source1_entity_id'], m['matched_entity_ids']):
        if not ids_str:
            continue
        ids = ids_str.split(',')
        if len(ids) != len(set(ids)):
            issues.append(f'{s1id}: duplicate ids within matched_entity_ids')
        for i in ids:
            if not (i.startswith('S2-') or i.startswith('S3-')):
                issues.append(f'{s1id}: invalid id prefix {i}'); break
            if i.startswith('S2-') and i not in valid_s2_ids:
                issues.append(f'{s1id}: {i} not in test_source2'); break
            if i.startswith('S3-') and i not in valid_s3_ids:
                issues.append(f'{s1id}: {i} not in test_source3'); break
        cand_ids = set((cand_map.get(s1id) or '').split(',')) if cand_map.get(s1id) else set()
        if not set(ids).issubset(cand_ids):
            issues.append(f'{s1id}: matched id(s) not present in its own candidate_pairs.tsv row')

    if issues:
        print(f'FAIL -- {len(issues)} issue(s), showing up to 20:')
        for x in issues[:20]:
            print(' -', x)
        return False
    print('PASS -- files look valid for submission.')
    return True

validate(match_path, cand_path, set(test_s1['entity_id']), valid_s2_ids, valid_s3_ids)


## If you still run out of RAM

In order of what to try:

1. **Runtime → Restart runtime** before re-running (clears memory from any earlier crashed attempt).
2. Lower `CHUNK_SIZE` (e.g. to 2,000–5,000) — this is the single biggest lever on peak memory now.
3. Lower `MAX_TRAIN_S1` (e.g. to 50,000–80,000) — training quality degrades gracefully, it doesn't fall off a cliff.
4. Runtime → Change runtime type → **High-RAM** (Colab Pro/Pro+), or run this on Kaggle Notebooks instead, which
   gives ~30GB RAM for free.
5. If a specific stage crashes (index build vs. training vs. test inference), tell me which `print()` line was
   the last one you saw — that pinpoints exactly which stage needs a smaller `CHUNK_SIZE`.

## Notes on pushing the score higher
- Check the blocking recall-ceiling print in cell 8 — that's your hard ceiling; if it's not near 1.0, the fix is
  wider blocking (raise `TOPK_CANDIDATES`, add more index keys), not the classifier.
- F0.5 is precision-heavy, so keep the tuned threshold high rather than lowering it to "catch more".
- Try per-country thresholds, richer address-component features, and TF-IDF cosine similarity as next steps.
